# Apartment Maintenance Dashboard

Explore the monthly maintenance workbook and visualize collection status. UTR values currently in the workbook are randomly generated samples, not transaction records.

In [1]:
%pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable
  Using cached streamlit-1.66.0-py3-none-any.whl.metadata (10 kB)
  Using cached plotly-6.9.0-py3-none-any.whl.metadata (9.0 kB)
  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached altair-6.3.0-py3-none-any.whl.metadata (11 kB)
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached pydeck-0.9.3-py2.py3-none-any.whl.metadata (4.2 kB)
  Using cached starlette-1.8.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached uvicorn-0.54.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached anyio-4.15.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached python_multipart-0.0.32-py3-none-any.whl.metadata (2.1 kB)
  Using cached itsdangerous-2.2.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached watchdog-6.0.0-py3-none-win_amd64.whl.metadata (44 kB)
  Using cached narwhals-2.27.1-py3-none-any.whl.metadata (15 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\HP\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px

workbook_path = Path('Monthly_Maintenance.xlsx')
if not workbook_path.exists():
    raise FileNotFoundError(f'Workbook not found: {workbook_path.resolve()}')

required_columns = [
    'Apartment No.',
    'Monthly Contribution (INR)',
    'Payment Date',
    'UPI UTR Number (Sample)',
    'Amount Paid (INR)',
]
header_probe = pd.read_excel(workbook_path, sheet_name='Monthly Maintenance', header=None, nrows=5)
header_row = next(
    (
        row_number
        for row_number, row in header_probe.iterrows()
        if set(required_columns).issubset({str(value).strip() for value in row.dropna()})
    ),
    None,
)
if header_row is None:
    raise ValueError('Could not find all required maintenance columns in the first five rows.')

ledger = pd.read_excel(
    workbook_path,
    sheet_name='Monthly Maintenance',
    header=header_row,
    converters={
        'UPI UTR Number (Sample)': lambda value: '' if pd.isna(value) else str(value).removesuffix('.0')
    },
)
ledger = ledger[required_columns].copy()
ledger['Apartment No.'] = pd.to_numeric(ledger['Apartment No.'], errors='coerce')
ledger = ledger.dropna(subset=['Apartment No.']).copy()
ledger['Apartment No.'] = ledger['Apartment No.'].astype('Int64').astype(str)
for column in ('Monthly Contribution (INR)', 'Amount Paid (INR)'):
    ledger[column] = pd.to_numeric(ledger[column], errors='coerce').fillna(0)
ledger['Payment Date'] = pd.to_datetime(ledger['Payment Date'], errors='coerce')
paid = ledger['Amount Paid (INR)']
due = ledger['Monthly Contribution (INR)']
ledger['Status'] = 'Unpaid'
ledger.loc[(paid > 0) & (paid < due), 'Status'] = 'Partial'
ledger.loc[paid >= due, 'Status'] = 'Paid'
ledger['Outstanding (INR)'] = (due - paid).clip(lower=0)
ledger.head()

In [ ]:
expected = ledger['Monthly Contribution (INR)'].sum()
collected = ledger['Amount Paid (INR)'].sum()
outstanding = ledger['Outstanding (INR)'].sum()
print(f'Expected: ₹{expected:,.0f}')
print(f'Collected: ₹{collected:,.0f}')
print(f'Outstanding: ₹{outstanding:,.0f}')
print(f'Apartments: {len(ledger)}')
ledger['Status'].value_counts().reindex(['Paid', 'Partial', 'Unpaid'], fill_value=0).rename('Apartments')

In [ ]:
status_order = ['Paid', 'Partial', 'Unpaid']
status_counts = (
    ledger['Status'].value_counts()
    .reindex(status_order, fill_value=0)
    .rename_axis('Status')
    .reset_index(name='Apartments')
)
status_chart = px.bar(
    status_counts,
    x='Status',
    y='Apartments',
    color='Status',
    text='Apartments',
    category_orders={'Status': status_order},
    color_discrete_map={'Paid': '#26745f', 'Partial': '#d99a2b', 'Unpaid': '#c85f48'},
)
status_chart.update_layout(showlegend=False, title='Apartments by payment status')
status_chart.show()

In [ ]:
balance_chart = px.bar(
    ledger.sort_values('Apartment No.'),
    x=['Amount Paid (INR)', 'Outstanding (INR)'],
    y='Apartment No.',
    orientation='h',
    barmode='stack',
    color_discrete_map={'Amount Paid (INR)': '#26745f', 'Outstanding (INR)': '#edc46e'},
    labels={'value': 'Amount (INR)', 'variable': '', 'Apartment No.': 'Apartment'},
)
balance_chart.update_layout(title='Paid and outstanding by apartment', xaxis_tickprefix='₹')
balance_chart.show()

In [ ]:
ledger[[
    'Apartment No.',
    'Monthly Contribution (INR)',
    'Payment Date',
    'UPI UTR Number (Sample)',
    'Amount Paid (INR)',
    'Status',
]].sort_values('Apartment No.')

## Run Locally

1. Keep `app.py`, `Monthly_Maintenance.xlsx`, and `requirements.txt` together in the project folder.
2. Open a terminal in that folder, install dependencies with `pip install -r requirements.txt`, then start the dashboard with `streamlit run app.py`.
3. Open the local URL printed in the terminal, usually `http://localhost:8501`.

## Deploy with Your GitHub Account

### 1. Create a GitHub repository

1. Sign in to [GitHub](https://github.com).
2. Select **New repository**, give it a name such as `apartment-maintenance-dashboard`, and create it. Choose **Public** for the simplest Community Cloud setup. A private repository can also be used if you grant Streamlit access to it.
3. In the repository, choose **Add file** > **Upload files**. Upload `app.py`, `requirements.txt`, and `Monthly_Maintenance.xlsx`. You may also include `Maintenance_Dashboard.ipynb`; do not upload the `.venv` folder.
4. Select **Commit changes** to save the files to the repository's `main` branch.

### 2. Deploy on Streamlit Community Cloud

1. Open [Streamlit Community Cloud](https://share.streamlit.io) and choose **Sign in with GitHub**. Authorize access to the repository you created.
2. Choose **Create app**, then select your repository and the `main` branch.
3. Set **Main file path** to `app.py`, then choose **Deploy**.
4. When the app finishes building, open the generated `*.streamlit.app` URL. That URL can be shared for web access.

### 3. Publish updates

Edit the files in your GitHub repository and commit the changes. Community Cloud will redeploy the app from the selected branch.

**Before deployment:** this dashboard displays workbook data, including the UTR column. The current UTRs are random samples; remove them or replace them with non-sensitive values before publishing real payment information. Anyone who can access the public app may see its data.